# Metal, dipole, and PML

This tutorial models a $y$-oriented electric point dipole above a metal substrate. It follows the corresponding [`metal_dipole.py` example](https://github.com/matt8s/fmmax/blob/main/examples/metal_dipole.py) and illustrates four related FMMAX workflows: representing lateral absorbing layers as anisotropic media, inserting a source inside a layer stack, solving for the amplitudes within each layer, and reconstructing the fields on an $x$–$z$ cross-section.

The calculation uses a finite square computational cell in the $x$ and $y$ directions. The Fourier expansion imposes periodicity on this cell, while uniaxial perfectly matched layers (PMLs) near its lateral edges attenuate fields before they reach the periodic boundaries. The layer sequence along $z$ is

```text
upper ambient
──────────────  dipole source plane
ambient gap
──────────────
metal substrate
```

The lateral PML regions extend through the ambient and metal layers. Because the PML is represented on a finite grid with a truncated Fourier basis, it should be understood as a practical absorber rather than an exactly reflection-free boundary.

## Simulation parameters

All lengths below are in micrometres (µm). The material parameters specify the complex scalar permittivities before the PML transformation. The three thicknesses define the displayed stack along $z$: ambient above the source, an ambient gap between the source and metal, and the depth reconstructed inside the metal substrate. `pitch` sets both lateral dimensions of the computational cell. `grid_shape` samples the material tensors in the $x$–$y$ plane, whereas `grid_spacing_fields` controls the later field sampling along $z$. The Fourier resolution is set separately by `approximate_num_terms`, and `pml_params` gives the number of grid points occupied by the PML at each $x$ and $y$ edge.

In [1]:
from fmmax import pml

# The permittivity of the ambient.
permittivity_ambient = (1.0 + 0.0j)
# The permittivity of the metal.
permittivity_metal = (-7.632 + 0.731j)
# The distance between the dipole and the metal.
thickness_dipole_metal_gap = 1.0
# The thickness of the ambient above the dipole.
thickness_ambient = 2.0
# The thickness of the metal layer.
thickness_metal = 0.1
# The x- and y-extent of the unit cell.
pitch = 5.0
# The shape of the grid used to represent permittivities and permeabilities.
grid_shape = (400, 400)
# The spacing of grid points for calculated fields.
grid_spacing_fields = 0.01
# The wavelength of dipole emission.
wavelength = 0.63
# The approximate number of terms used in the plane wave expansion of the fields.
approximate_num_terms = 1200
# Parameters specifying the extent and strength of the perfectly matched layers.
pml_params = pml.PMLParams(num_x=50, num_y=50)

## Build the periodic Fourier basis

The primitive lattice vectors define a square cell of width `pitch`. FMMAX expands the fields in plane waves associated with this periodic cell; circular truncation retains reciprocal-lattice vectors within an approximately circular region until the requested number of terms is reached.

Setting the in-plane Bloch wavevector to zero applies no additional phase from one cell to the next. The resulting fields are periodic in $x$ and $y$. The lateral PML bands make this periodic representation useful for an effectively isolated source by attenuating radiation near the cell edges and reducing coupling through the periodic boundary.

In [2]:
from jax import numpy as jnp
from fmmax import basis

primitive_lattice_vectors = basis.LatticeVectors(
    u=pitch * basis.X, v=pitch * basis.Y
)
expansion = basis.generate_expansion(
    primitive_lattice_vectors=primitive_lattice_vectors,
    approximate_num_terms=approximate_num_terms,
    truncation=basis.Truncation.CIRCULAR,
)
in_plane_wavevector = jnp.zeros((2,))

## Apply the lateral PML

A uniaxial PML can be expressed as a coordinate transformation that changes an initially scalar material into anisotropic permittivity and permeability tensors. `apply_uniaxial_pml()` constructs the five tensor components used by FMMAX—$xx$, $xy$, $yx$, $yy$, and $zz$—on the real-space grid. It is applied separately to the ambient and metal so that the absorbing bands retain the appropriate underlying material in every layer.

This PML construction assumes that the primitive lattice vectors are aligned with the Cartesian $x$ and $y$ axes, as they are here. It acts only near the $x$ and $y$ edges of the finite periodic cell. No PML is added along $z$; propagation in that direction is represented by the eigenmodes of the individual layers and assembled through the scattering formulation. In a discretized calculation with finitely many Fourier terms, some residual numerical reflection can remain.

In [3]:
permittivities_ambient_pml, permeabilities_ambient_pml = pml.apply_uniaxial_pml(
    permittivity=jnp.full(grid_shape, permittivity_ambient),
    pml_params=pml_params,
)
permittivities_metal_pml, permeabilities_metal_pml = pml.apply_uniaxial_pml(
    permittivity=jnp.full(grid_shape, permittivity_metal),
    pml_params=pml_params,
)

## Solve the anisotropic layer eigenproblems

Each layer is represented by its plane-wave eigenmodes. Because the PML transformation produces anisotropic permittivity and permeability tensors, the appropriate solver is `fmm.eigensolve_general_anisotropic_media()` rather than an isotropic eigensolver.

The ambient and metal calculations use the same wavelength, Bloch wavevector, lattice, Fourier expansion, and formulation. `functools.partial()` binds these arguments once, so each call to `eigensolve` only needs the material tensors. Here we use the FFT formulation; `vector_field_source=None` selects the default vector-field source.

In [4]:
from fmmax import fmm
import functools

eigensolve = functools.partial(
    fmm.eigensolve_general_anisotropic_media,
    wavelength=jnp.asarray(wavelength),
    in_plane_wavevector=in_plane_wavevector,
    primitive_lattice_vectors=primitive_lattice_vectors,
    expansion=expansion,
    formulation=fmm.Formulation.FFT,
    vector_field_source=None,  # Automatically choose the vector field source.
)

Calling the configured eigensolver with each set of PML-transformed tensor components gives two layer solutions. `solve_result_ambient` is reused for both ambient portions of the stack: the region above the source and the gap below it. `solve_result_metal` describes the metal substrate. The finite metal thickness used later sets the depth shown in the reconstructed field; there is no second metal interface in this model. Keeping these results separate provides the modal bases, propagation constants, and material operators needed to assemble the source-side substacks.

In [5]:
solve_result_ambient = eigensolve(
    permittivity_xx=permittivities_ambient_pml[0],
    permittivity_xy=permittivities_ambient_pml[1],
    permittivity_yx=permittivities_ambient_pml[2],
    permittivity_yy=permittivities_ambient_pml[3],
    permittivity_zz=permittivities_ambient_pml[4],
    permeability_xx=permeabilities_ambient_pml[0],
    permeability_xy=permeabilities_ambient_pml[1],
    permeability_yx=permeabilities_ambient_pml[2],
    permeability_yy=permeabilities_ambient_pml[3],
    permeability_zz=permeabilities_ambient_pml[4],
)
solve_result_metal = eigensolve(
    permittivity_xx=permittivities_metal_pml[0],
    permittivity_xy=permittivities_metal_pml[1],
    permittivity_yx=permittivities_metal_pml[2],
    permittivity_yy=permittivities_metal_pml[3],
    permittivity_zz=permittivities_metal_pml[4],
    permeability_xx=permeabilities_metal_pml[0],
    permeability_xy=permeabilities_metal_pml[1],
    permeability_yx=permeabilities_metal_pml[2],
    permeability_yy=permeabilities_metal_pml[3],
    permeability_zz=permeabilities_metal_pml[4],
)

## Form scattering matrices on both sides of the source

A scattering matrix relates the forward- and backward-going modal amplitudes at the boundaries of a layer or substack. Here the source divides the geometry into two disjoint substacks:

- the before-source substack contains the upper ambient layer;
- the after-source substack contains the ambient gap followed by the metal layer.

Keeping these substacks separate allows the dipole to be inserted explicitly at their common interface. `stack_s_matrices_interior()` returns the cumulative scattering matrices before and after every layer, rather than only the response of the complete stack. Those interior matrices will later be used to recover the modal amplitudes inside each layer for field reconstruction.

In [6]:
from fmmax import scattering

# Compute interior scattering matrices to enable field calculations.
s_matrices_interior_before_source = scattering.stack_s_matrices_interior(
    layer_solve_results=[solve_result_ambient],
    layer_thicknesses=[jnp.asarray(thickness_ambient)],
)
s_matrices_interior_after_source = scattering.stack_s_matrices_interior(
    layer_solve_results=[solve_result_ambient, solve_result_metal],
    layer_thicknesses=[
        jnp.asarray(thickness_dipole_metal_gap),
        jnp.asarray(thickness_metal),
    ],
)

## Represent the embedded dipole

`sources.dirac_delta_source()` computes the plane-wave coefficients of a point source at the center of the plotted cell, $(x,y)=(\mathrm{pitch}/2,\mathrm{pitch}/2)$. This cell is periodic, so the Fourier representation is periodic as well; the lateral PML reduces interaction with neighboring copies.

At this stage, `dipole` contains only the spatial Fourier coefficients. The electric-current orientation is assigned later by placing these coefficients in the $y$ component and setting the $x$ and $z$ components to zero. The source is embedded because it lies between two substacks and launches fields toward both of them. With a finite Fourier expansion, the numerical representation of the delta function is necessarily band-limited.

In [7]:
from fmmax import sources

# Generate the Fourier representation of a point dipole.
dipole = sources.dirac_delta_source(
    location=jnp.asarray([[pitch / 2, pitch / 2]]),
    in_plane_wavevector=in_plane_wavevector,
    primitive_lattice_vectors=primitive_lattice_vectors,
    expansion=expansion,
)

The source calculation needs one scattering matrix for the complete substack on each side of the dipole. For every layer, `stack_s_matrices_interior()` returns a pair containing the cumulative before- and after-layer matrices. Selecting the first member of the upper substack's only entry gives its complete before-source matrix. Selecting the same member at the final entry of the lower substack gives the cumulative matrix for the ambient gap and metal together.

In [8]:
s_matrix_before_source = s_matrices_interior_before_source[0][0]
s_matrix_after_source = s_matrices_interior_after_source[-1][0]

## Convert the source current into modal amplitudes

`sources.amplitudes_for_source()` converts the Fourier coefficients of the electric current into eigenmode amplitudes while accounting for the structures on both sides of the source. Only `jy` is nonzero, so this is a $y$-polarized electric dipole.

The function returns amplitudes at several substack boundaries. The two retained here are the backward-going amplitude immediately before the source and the forward-going amplitude immediately after it. They describe excitation away from the source into the upper and lower substacks and provide the boundary data needed to solve for fields throughout the stack.

In [9]:
# Compute backward eigenmode amplitudes at the end of the layer before the
# source, and the forward amplitudes the start of the layer after the source.
(
    _,
    _,
    bwd_amplitude_before_end,
    fwd_amplitude_after_start,
    _,
    _,
) = sources.amplitudes_for_source(
    jx=jnp.zeros_like(dipole),
    jy=dipole, # Only excite the y polarization.
    jz=jnp.zeros_like(dipole),
    s_matrix_before_source=s_matrix_before_source,
    s_matrix_after_source=s_matrix_after_source,
)

## Recover amplitudes inside every layer

The amplitudes adjacent to the source do not by themselves provide the field at arbitrary depths. `fields.stack_amplitudes_interior_with_source()` combines them with the full sets of interior scattering matrices to determine the forward amplitude at the start and the backward amplitude at the end of every layer.

The returned sequence follows the physical layer order: upper ambient, ambient gap, and the reconstructed portion of the metal substrate. Retaining these layer-resolved amplitudes is essential for reconstructing fields inside the structure; a single scattering matrix for the complete stack would describe only its boundary response.

In [10]:
from fmmax import fields

# Solve for the eigenmode amplitudes in every layer of the stack.
amplitudes_interior = fields.stack_amplitudes_interior_with_source(
    s_matrices_interior_before_source=s_matrices_interior_before_source,
    s_matrices_interior_after_source=s_matrices_interior_after_source,
    backward_amplitude_before_end=bwd_amplitude_before_end,
    forward_amplitude_after_start=fwd_amplitude_after_start,
)

## Reconstruct an $x$–$z$ field slice

The quantities computed so far are modal coefficients. `fields.stack_fields_3d_on_coordinates()` propagates those modes to the requested $z$ positions and evaluates their plane-wave expansions at specified lateral coordinates.

To obtain a cross-section through the dipole, `x` spans one full period while `y` remains fixed at the cell midpoint. The layer solve results and thicknesses repeat the ambient solution on both sides of the source and then append the metal solution. `layer_znum` sets the number of samples in each layer from `grid_spacing_fields`. The function returns all Cartesian components of the electric and magnetic fields together with the corresponding coordinates.

In [11]:
# Coordinates where fields are to be evaluated.
x = jnp.linspace(0, pitch, grid_shape[0])
y = jnp.ones_like(x) * pitch / 2
(ex, ey, ez), (hx, hy, hz), (x, y, z) = fields.stack_fields_3d_on_coordinates(
    amplitudes_interior=amplitudes_interior,
    layer_solve_results=[
        solve_result_ambient,
        solve_result_ambient,
        solve_result_metal,
    ],
    layer_thicknesses=[
        jnp.asarray(thickness_ambient),
        jnp.asarray(thickness_dipole_metal_gap),
        jnp.asarray(thickness_metal),
    ],
    layer_znum=[
        int(thickness_ambient / grid_spacing_fields),
        int(thickness_dipole_metal_gap / grid_spacing_fields),
        int(thickness_metal / grid_spacing_fields),
    ],
    x=x,
    y=y,
)

## Plot the electric-field magnitude

The complex Cartesian components are combined as

$$
|\mathbf{E}|=\sqrt{|E_x|^2+|E_y|^2+|E_z|^2}.
$$

The selected two-dimensional array is plotted on the $x$–$z$ mesh. Clipping the color scale at the 99.8th percentile prevents the largest values near the point-source representation from overwhelming the rest of the image; it changes only the visualization, not the calculated field. Equal axis scaling preserves the geometry, and reversing the displayed $z$ axis places the upper ambient region above the metal.

In [ ]:
import matplotlib.pyplot as plt

field_3d = jnp.sqrt(jnp.abs(ex) ** 2 + jnp.abs(ey) ** 2 + jnp.abs(ez) ** 2)
field_plot = field_3d[:, :, 0]

xplot, zplot = jnp.meshgrid(x, z, indexing="ij")

plt.figure(figsize=(float(jnp.amax(xplot)), float(jnp.amax(zplot))), dpi=80)
ax = plt.subplot(111)
im = ax.pcolormesh(xplot, zplot, field_plot, shading="nearest", cmap="magma")

# Clip the color scale to the 99.8th percentile of the field.
clip_percentile = 99.8
clipval = float(jnp.percentile(field_plot, clip_percentile))
im.set_clim((0, clipval))

ax.axis("equal")
ax.axis("off")
ax.set_ylim(ax.get_ylim()[::-1])

plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
plt.show()

When you run the plotting cell, the field should decrease laterally as it enters the PML bands near the sides of the cell, with the metal region visible beneath the ambient gap. This is a qualitative check that the reconstructed field follows the intended geometry and lateral attenuation; the image alone does not provide a quantitative measure of residual reflection.

The apparent finite width of the source comes from representing a Dirac delta with a truncated plane-wave basis. Increasing the number of Fourier terms can sharpen this representation, at the cost of a larger eigenproblem. See the [periodic dipole tutorial](./dipoles.md) for a more direct introduction to Fourier-domain dipole sources.